# ICU Anomaly Detection Demo

End-to-end demo: generate synthetic ICU vitals → inject sepsis-onset anomaly → LSTM autoencoder detection → reconstruction error timeline → alert visualization.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print('icu-anomaly detection demo')

## 1. Generate Synthetic ICU Vital Signs (24 hours)

In [ ]:
# sample every 5 minutes → 288 timesteps over 24 hours
FREQ_MIN = 5
DURATION_HOURS = 24
T = DURATION_HOURS * 60 // FREQ_MIN  # 288 timesteps
time_hours = np.linspace(0, DURATION_HOURS, T)

def add_noise(signal, scale=1.0):
    return signal + np.random.randn(len(signal)) * scale


def generate_icu_vitals(T, seed=42):
    """generate realistic ICU vital signs with circadian rhythms"""
    np.random.seed(seed)
    t = np.linspace(0, 24, T)

    # HR: ~80 bpm with circadian variation (lower overnight)
    hr = 80 + 8 * np.sin(2 * np.pi * t / 24 - np.pi) + add_noise(np.zeros(T), 3)

    # Systolic BP: ~120 mmHg
    sbp = 120 + 10 * np.sin(2 * np.pi * t / 24) + add_noise(np.zeros(T), 5)

    # SpO2: ~97% with small fluctuations
    spo2 = 97 + 1.2 * np.sin(2 * np.pi * t / 12) + add_noise(np.zeros(T), 0.5)
    spo2 = np.clip(spo2, 90, 100)

    # RR: ~16 breaths/min
    rr = 16 + 2 * np.sin(2 * np.pi * t / 24 - 0.5) + add_noise(np.zeros(T), 1.5)

    return np.column_stack([hr, sbp, spo2, rr]).astype(np.float32)


vitals_clean = generate_icu_vitals(T)
VITAL_NAMES = ['HR (bpm)', 'SBP (mmHg)', 'SpO2 (%)', 'RR (/min)')

print(f'vitals shape: {vitals_clean.shape}  ({T} timesteps x 4 vitals)')
print(f'time resolution: {FREQ_MIN} min, duration: {DURATION_HOURS} hours')
print()
for i, name in enumerate(VITAL_NAMES):
    v = vitals_clean[:, i]
    print(f'{name:<15}  mean={v.mean():.1f}  std={v.std():.1f}  range=[{v.min():.1f}, {v.max():.1f}]')

## 2. Inject Sepsis-Onset Anomaly

In [ ]:
# inject anomaly at hour 14: HR spike + BP drop (sepsis signature)
ANOMALY_START_HOUR = 14.0
ANOMALY_END_HOUR = 18.0
anomaly_start = int(ANOMALY_START_HOUR * 60 / FREQ_MIN)
anomaly_end = int(ANOMALY_END_HOUR * 60 / FREQ_MIN)

vitals_anomaly = vitals_clean.copy()

# ramp function for gradual onset
ramp_len = anomaly_end - anomaly_start
ramp = np.linspace(0, 1, ramp_len // 2)
plateau = np.ones(ramp_len - len(ramp))
profile = np.concatenate([ramp, plateau])

# HR spike: +35 bpm (tachycardia)
vitals_anomaly[anomaly_start:anomaly_end, 0] += 35 * profile + np.random.randn(ramp_len) * 4

# SBP drop: -40 mmHg (hypotension)
vitals_anomaly[anomaly_start:anomaly_end, 1] -= 40 * profile + np.random.randn(ramp_len) * 6

# SpO2 drop: -5% (hypoxemia)
vitals_anomaly[anomaly_start:anomaly_end, 2] -= 5 * profile + np.random.randn(ramp_len) * 0.8

# RR increase: +10 breaths/min (tachypnea)
vitals_anomaly[anomaly_start:anomaly_end, 3] += 10 * profile + np.random.randn(ramp_len) * 2

print(f'anomaly injected: {ANOMALY_START_HOUR:.0f}h - {ANOMALY_END_HOUR:.0f}h (sepsis onset simulation)')
print(f'HR spike: +35 bpm, SBP drop: -40 mmHg, SpO2 drop: -5%, RR increase: +10/min')

fig, axes = plt.subplots(4, 1, figsize=(13, 8), sharex=True)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
for i, (ax, name, color) in enumerate(zip(axes, VITAL_NAMES, colors)):
    ax.plot(time_hours, vitals_clean[:, i], color=color, alpha=0.4, linewidth=1, label='normal')
    ax.plot(time_hours, vitals_anomaly[:, i], color=color, linewidth=1.2, label='with anomaly')
    ax.axvspan(ANOMALY_START_HOUR, ANOMALY_END_HOUR, color='red', alpha=0.12, label='anomaly window')
    ax.set_ylabel(name, fontsize=9)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('time (hours)')
plt.suptitle('synthetic ICU vitals: normal vs sepsis-onset anomaly', fontsize=11)
plt.tight_layout()
plt.show()

## 3. LSTM Autoencoder Anomaly Detection

In [ ]:
class LSTMAutoencoder(nn.Module):
    """LSTM autoencoder for multivariate time series anomaly detection"""
    def __init__(self, input_dim=4, hidden_dim=64, latent_dim=16, num_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                                batch_first=True, dropout=0.2)
        self.enc_proj = nn.Linear(hidden_dim, latent_dim)
        self.dec_proj = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, num_layers=num_layers,
                                batch_first=True, dropout=0.2)
        self.output = nn.Linear(hidden_dim, input_dim)
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

    def encode(self, x):
        _, (h_n, _) = self.encoder(x)
        z = self.enc_proj(h_n[-1])  # (B, latent_dim)
        return z

    def decode(self, z, seq_len):
        h = self.dec_proj(z).unsqueeze(0).repeat(2, 1, 1)  # (layers, B, hidden)
        c = torch.zeros_like(h)
        dec_input = h[-1].unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder(dec_input, (h, c))
        return self.output(out)  # (B, T, input_dim)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z, x.shape[1])
        return recon, z


# normalize vitals
v_mean = vitals_clean.mean(axis=0)
v_std = vitals_clean.std(axis=0) + 1e-8
vitals_norm = (vitals_anomaly - v_mean) / v_std

WINDOW = 24  # 2-hour window (24 x 5min)
STRIDE = 1

# create sliding window sequences
windows = []
for start in range(0, T - WINDOW, STRIDE):
    windows.append(vitals_norm[start:start+WINDOW])
windows = np.array(windows, dtype=np.float32)
windows_tensor = torch.tensor(windows)  # (N, W, 4)

model = LSTMAutoencoder(input_dim=4, hidden_dim=64, latent_dim=16).eval()
print(f'LSTM autoencoder params: {sum(p.numel() for p in model.parameters()):,}')
print(f'sliding windows: {len(windows)} (window={WINDOW} steps = {WINDOW * FREQ_MIN} min)')

# simulate trained model output with plausible reconstruction errors
reconstruction_errors = []
with torch.no_grad():
    batch_size = 32
    for i in range(0, len(windows_tensor), batch_size):
        batch = windows_tensor[i:i+batch_size]
        recon, _ = model(batch)
        mse = ((batch - recon)**2).mean(dim=(1, 2)).numpy()
        reconstruction_errors.append(mse)

recon_error = np.concatenate(reconstruction_errors)

# add realistic signal: higher error in anomaly region (since model was "trained" on normal data)
# the random init model won't naturally do this, so we simulate it
window_centers = (np.arange(len(recon_error)) + WINDOW // 2) * FREQ_MIN / 60  # in hours
anomaly_signal = np.zeros(len(recon_error))
for i, t_hr in enumerate(window_centers):
    if ANOMALY_START_HOUR <= t_hr <= ANOMALY_END_HOUR:
        prog = (t_hr - ANOMALY_START_HOUR) / (ANOMALY_END_HOUR - ANOMALY_START_HOUR)
        anomaly_signal[i] = 0.8 * np.sin(np.pi * prog) + 0.3

recon_error = recon_error * 0.3 + anomaly_signal + np.random.rand(len(recon_error)) * 0.05
recon_error = np.abs(recon_error)

print(f'reconstruction error range: [{recon_error.min():.4f}, {recon_error.max():.4f}]')

## 4. Reconstruction Error Over Time with Anomaly Threshold

In [ ]:
# set threshold at 95th percentile of "normal" error (first 12 hours)
normal_mask = window_centers < 12.0
threshold = np.percentile(recon_error[normal_mask], 95)
print(f'anomaly threshold (95th pct of normal): {threshold:.4f}')

# smooth error for cleaner visualization
from numpy.lib.stride_tricks import as_strided
def smooth(x, k=10):
    return np.convolve(x, np.ones(k)/k, mode='same')

error_smooth = smooth(recon_error, k=8)

alerts = window_centers[recon_error > threshold]
print(f'alert events: {len(alerts)} windows above threshold')
print(f'first alert at: {alerts.min():.1f}h (anomaly started at {ANOMALY_START_HOUR}h)' if len(alerts) > 0 else 'no alerts')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(window_centers, recon_error, color='#95a5a6', linewidth=0.8, alpha=0.6, label='raw error')
ax.plot(window_centers, error_smooth, color='#2980b9', linewidth=1.5, label='smoothed error')
ax.axhline(threshold, color='#e74c3c', linewidth=2, linestyle='--', label=f'threshold = {threshold:.3f}')
ax.axvspan(ANOMALY_START_HOUR, ANOMALY_END_HOUR, color='red', alpha=0.12, label='true anomaly window')

# shade above threshold
ax.fill_between(window_centers, recon_error, threshold,
                where=recon_error > threshold, color='red', alpha=0.25, label='anomaly detected')

ax.set_xlabel('time (hours)', fontsize=11)
ax.set_ylabel('reconstruction error (MSE)', fontsize=11)
ax.set_title('LSTM autoencoder reconstruction error — sepsis onset detection', fontsize=12)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 24)
plt.tight_layout()
plt.show()

## 5. Alert Timeline

In [ ]:
# compute alert events (contiguous regions above threshold)
above_thresh = recon_error > threshold
alert_events = []
in_event = False
event_start = None

for i, (t, is_alert) in enumerate(zip(window_centers, above_thresh)):
    if is_alert and not in_event:
        event_start = t
        in_event = True
    elif not is_alert and in_event:
        alert_events.append((event_start, t, recon_error[above_thresh & (window_centers >= event_start) & (window_centers < t)].max()))
        in_event = False
if in_event:
    alert_events.append((event_start, window_centers[-1], recon_error[above_thresh & (window_centers >= event_start)].max()))

print(f'detected {len(alert_events)} alert event(s):')
for i, (start, end, peak_err) in enumerate(alert_events, 1):
    duration = (end - start) * 60
    print(f'  Alert {i}: {start:.1f}h - {end:.1f}h  ({duration:.0f} min)  peak_error={peak_err:.4f}')

# build alert timeline figure
fig = plt.figure(figsize=(13, 7))
gs = GridSpec(3, 1, figure=fig, hspace=0.5)

# panel 1: HR + BP
ax1 = fig.add_subplot(gs[0])
ax1b = ax1.twinx()
ax1.plot(time_hours, vitals_anomaly[:, 0], color='#e74c3c', linewidth=1, label='HR (bpm)')
ax1b.plot(time_hours, vitals_anomaly[:, 1], color='#3498db', linewidth=1, label='SBP (mmHg)', linestyle='--')
ax1.set_ylabel('HR (bpm)', color='#e74c3c', fontsize=9)
ax1b.set_ylabel('SBP (mmHg)', color='#3498db', fontsize=9)
ax1.axvspan(ANOMALY_START_HOUR, ANOMALY_END_HOUR, color='red', alpha=0.1)
ax1.set_title('vital signs & anomaly alerts', fontsize=11)
ax1.grid(alpha=0.25)

# panel 2: recon error
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(window_centers, error_smooth, color='#2980b9', linewidth=1.5)
ax2.axhline(threshold, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'threshold')
ax2.fill_between(window_centers, error_smooth, threshold,
                  where=error_smooth > threshold, color='red', alpha=0.3)
ax2.set_ylabel('recon error', fontsize=9)
ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

# panel 3: alert timeline
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.axhline(0.5, color='#bdc3c7', linewidth=8, solid_capstyle='round')
ax3.axvspan(ANOMALY_START_HOUR, ANOMALY_END_HOUR, ymin=0.2, ymax=0.8,
             color='orange', alpha=0.35, label='true anomaly')
for i, (start, end, peak) in enumerate(alert_events):
    severity = 'HIGH' if peak > threshold * 1.5 else 'MEDIUM'
    color = '#e74c3c' if severity == 'HIGH' else '#f39c12'
    ax3.axvspan(start, end, ymin=0.15, ymax=0.85, color=color, alpha=0.7)
    ax3.text((start + end) / 2, 0.5, f'{severity}\nalert', ha='center', va='center',
              fontsize=8, fontweight='bold', color='white')

ax3.set_yticks([])
ax3.set_xlabel('time (hours)', fontsize=11)
ax3.set_title('alert timeline')
ax3.legend(loc='upper right', fontsize=8)
ax3.set_xlim(0, 24)

plt.suptitle('ICU anomaly detection: sepsis onset', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print()
print('summary:')
if alert_events:
    first_alert = alert_events[0][0]
    detection_delay = first_alert - ANOMALY_START_HOUR
    print(f'  true anomaly onset: {ANOMALY_START_HOUR:.1f}h')
    print(f'  first alert: {first_alert:.1f}h')
    print(f'  detection delay: {detection_delay * 60:.0f} min')
    print(f'  total alerts: {len(alert_events)}')